In [ ]:
import pandas as pd
from data import CollectionAccessor, ImageHandler

In [ ]:
image_folder = "./DMG/images"
image_handler = ImageHandler(image_folder=image_folder, keep_prefix=False)

time_stamp, pub_file, priv_file = CollectionAccessor.get_latest_dump("./DMG/dumps")


dmg_meta = dict(name="Design Museum Gent (public & private)", id_="DMG_"+time_stamp,
                creation_timestamp=time_stamp)
dmg = CollectionAccessor.get_DMG(pub_path=pub_file, #get_latest("./data/dumps", contains="public"),
                                     priv_path=priv_file, #get_latest("./data/dumps", contains="private"),
                                     rights_path="./DMG/rights.csv",
                                     image_handler=image_handler,
                                     **dmg_meta)


In [ ]:
print("\n".join(dmg.columns))

In [ ]:
dmg.coll.filter("2009-0089")

---
# MKG

# TODO

 - translate string constants into German:
   - get_texts
   - get_presentation_records
   - human_readable_dates
 - 

In [ ]:
from data import ImageHandler, CollectionAccessor

image_folder = "./MKG/images"
image_handler = ImageHandler("MKG", image_folder, keep_prefix=True)


time_stamp = "2025-06-05"
mkg_meta = dict(name="Museum Kunst & Gewerbe", id_="MKG_"+time_stamp,
                creation_timestamp=time_stamp, language="de")
mkg = CollectionAccessor.get_MKG(metadata_path="./MKG/dumps/extraction_v0_1.csv",
                                image_handler=image_handler,
                                **mkg_meta)

In [ ]:
mkg.image_path.notna().sum()

In [ ]:
mkg.image_path.dropna()

---

# ImageHandlers

In [ ]:
import os
f = "~/Desktop/SerendipitySearch/searcher_backend/second_web_backend/data/data.py"
print(os.path.basename(f))
print(os.path.splitext(f))

print()
print(os.path.splitext(os.path.basename(f)))

In [ ]:
image_handler.object_number_from_path(mkg.reset_index().object_number)

In [ ]:
mkg.image_path.dropna()

---

# MKG image downloads

In [ ]:
from tqdm import tqdm 

import requests as req
import pandas as pd
from time import sleep
import os

BASE_DIR = "./MKG/dumps"
IMG_DIR = "./MKG/images"


# if True:
df = pd.read_csv(BASE_DIR + "/extraction_v0_1.csv")
    
cur_df = df.dropna(subset="img_url")
    
# for i, r in tqdm(cur_df.iterrows(), total=len(cur_df)):
    # if r.img_url:


In [ ]:
cur_df.img_url.sample(1).iloc[0]

---
### dominant colour

In [ ]:
# from imagedominantcolor import DominantColor
# file_path = "/home/valentin/Pictures/DSC_0195.JPG"
# dominantcolor = DominantColor(file_path)
# dominantcolor.dominant_color
# dominantcolor.rgb

from colorthief import ColorThief

color_thief = ColorThief('/home/valentin/Downloads/IMG_6211.jpg')




In [ ]:
dominant_color = color_thief.get_color(quality=1)

In [ ]:
print(dominant_color)

import numpy as np
arr = np.asarray(color_thief.image)
w, h, col_dims = arr.shape
arr.sum(0).sum(0)/(w*h)

---
## resizing

In [ ]:
import os
from PIL import Image
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True


In [ ]:
# def resize(image_handle, new_size, new_path):
#     def resize(w, h):
#         fixed_size = new_size # CHANGED; WAS 1200
#         r = h/w 
#         if w >= h:
#             return (fixed_size, int(fixed_size*r))
#         else:
#             r = 1/r
#             return (int(fixed_size*r), fixed_size)
#     new_size = resize(*img_handle.size)
#     img_handle.thumbnail(new_size, Image.Resampling.LANCZOS)
#     if not os.path.isdir(new_path+row.prefix):
#         os.makedirs(new_path+row.prefix)
#     img_handle.save(new_path+row.raw, quality=80)
#     return new_size

def resize(image_handle, max_size):
    w, h = image_handle.size
    r = h/w 
    if w >= h:
        new_size = (max_size, int(max_size*r))
    else:
        r = 1/r
        new_size = (int(max_size*r), max_size)
    return image_handle.resize(new_size, Image.Resampling.LANCZOS)


In [ ]:
img_path = "/home/valentin/Pictures/test.jpg"
with Image.open(img_path) as img_handle:
    # new_img = img_handle.resize((100, 100))
    new_img = resize(img_handle, 600)

    new_img.save(img_path, quality=80)


In [ ]:
new_img

---
# using `image_info.csv`


new data structure for image (replace `image_path` key)

```
"image":
   {
      "path": "<some_path>",
      “thumb”: {
          "path": "<some_path>",
          "width": <some_number>,
          "height": <some_number>,      
      },
      "width": <some_number>,
      "height": <some_number>,
      …
   }
```

In [11]:
import pandas as pd
from data import ImageHandler, DMGImageHandler, CollectionAccessor

In [ ]:
i = pd.read_csv("./DMG/images/image_info.csv").set_index("object_number")

In [ ]:
f = pd.read_csv("./DMG/images/filenames.csv")


f.raw.apply(lambda p: i.path.str.endswith(p)).sum(0)

In [ ]:
primary_images = pd.read_csv("./DMG/primary_images.csv")
primary_images = primary_images.dropna(subset=["objectnummer", "naam beeld(en)"]).set_index("objectnummer")
primary_images = primary_images["naam beeld(en)"].fillna("").str.split(";").apply(lambda ls: ls[0])


primary_images

common = sorted(set(i.index) & set(primary_images.index))

primary_images.loc[common] + ".jpg"

In [ ]:
i.loc[common].filename.str.lower()#.apply(lambda s: s.endswith(primary_images.loc[common] + ".jpg"))

primary_images

---

In [2]:
image_handler = DMGImageHandler("./DMG/images/")

ih._obj

NameError: name 'DMGImageHandler' is not defined

In [3]:
DMG_DIR = "./DMG"
image_folder = DMG_DIR+"/images/"
image_handler = ImageHandler("DMG", image_folder=image_folder, keep_prefix=False)

time_stamp, pub_file, priv_file = CollectionAccessor.get_latest_dump(DMG_DIR+"/dumps")


dmg_meta = dict(name="Design Museum Gent (public & private)", id_="DMG_"+time_stamp,
                creation_timestamp=time_stamp, language="nl")

df = CollectionAccessor.get_DMG(pub_path=pub_file, #get_latest("./data/dumps", contains="public"),
                                     priv_path=priv_file, #get_latest("./data/dumps", contains="private"),
                                     rights_path=DMG_DIR+"/rights.csv",
                                     image_handler=image_handler,
                                     **dmg_meta)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24781/24781 [00:11<00:00, 2117.12it/s]


In [27]:
for i, r in df.fillna("").iterrows():
    print(r.index)

Index(['object_URI', 'title', 'description', 'objectname_URI',
       'objectname_label', 'subcollection_URI', 'subcollection_type',
       'subcollection_name', 'material_URI', 'material_label', 'part_label',
       'part_material_URI', 'part_material_label', 'creation_time',
       'creation_place_URI', 'creation_place_label', 'maker_URI',
       'maker_label', 'technique_URI', 'technique_label', 'coin_time',
       'coin_place_URI', 'coin_place_label', 'coiner_URI', 'coiner_label',
       'acquisition_time', 'is_public', 'Unnamed: 0', 'rights', 'attribution',
       'width', 'height', 'thumb_width', 'thumb_height', 'filename', 'path',
       'thumb_path', 'dominant_R', 'dominant_G', 'dominant_B', 'time',
       'sort_rank'],
      dtype='object')
Index(['object_URI', 'title', 'description', 'objectname_URI',
       'objectname_label', 'subcollection_URI', 'subcollection_type',
       'subcollection_name', 'material_URI', 'material_label', 'part_label',
       'part_material_URI', 'p

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Index(['object_URI', 'title', 'description', 'objectname_URI',
       'objectname_label', 'subcollection_URI', 'subcollection_type',
       'subcollection_name', 'material_URI', 'material_label', 'part_label',
       'part_material_URI', 'part_material_label', 'creation_time',
       'creation_place_URI', 'creation_place_label', 'maker_URI',
       'maker_label', 'technique_URI', 'technique_label', 'coin_time',
       'coin_place_URI', 'coin_place_label', 'coiner_URI', 'coiner_label',
       'acquisition_time', 'is_public', 'Unnamed: 0', 'rights', 'attribution',
       'width', 'height', 'thumb_width', 'thumb_height', 'filename', 'path',
       'thumb_path', 'dominant_R', 'dominant_G', 'dominant_B', 'time',
       'sort_rank'],
      dtype='object')
Index(['object_URI', 'title', 'description', 'objectname_URI',
       'objectname_label', 'subcollection_URI', 'subcollection_type',
       'subcollection_name', 'material_URI', 'material_label', 'part_label',
       'part_material_URI', 'p

In [16]:
# df.join(image_handler._obj, how="left")

ih_index = image_handler._obj.index

from_path = DMGImageHandler.object_number_from_path(image_handler._obj.path)

In [19]:
len(set(df.index) & set(ih_index))

399

In [31]:
sub = df.dropna(subset="path")
sub.index.str.contains("|".join(sub.index), regex=True)

"|".join(sub.index)

'0008|0195|0021|0070|0076|0073|0235|0042|0043|0196|0197|0199|0200|0201|0202|0203|0204|0205|0206|0207|0208|0210|0211|0212|0213|0214|0215|0216|0217|0218|0219|0220|0221|0222|0224|0225|0227|0228|0229|0230|0231|0233|0237|0239|0240|0244|0376|0383|0401|0104_0-3|0104_1-3|0104_2-3|0104_3-3|0040|0286|0288|0353|0354|0355|0356|0357|0358|0359|0360|0361|0362|0363|0364|0365|0366|0368|0369|0370|0371|0372|0373|0374|0375|0377|0378|0379|0380|0381|0382|0384|0385|0386|0387|0388|0389|0390|0391|0392|0393|0394|0395|0396|0397|0399|0402|0404|0405|0407|0408|0409|0410|0412|0413|0414|0415|0416|0417|0418|0419|0420|0422|0423|0424|0425|0426|0427|0428|0429|0055_1|0055_2|0003_3-3|0029_1-2|0029_2-2|0054|0113|0209|0223|0236|0245|0246|0247|0248|0249|0250|0252|0253|0254|0255|0257|0258|0259|0260|0262|0263|0265|0266|0267|0268|0269|0270|0271|0272|0273|0274|0275|0279|0280|0281|0282|0283|0284|0285|0287|0289|0290|0291|0292|0293|0295|0296|0297|0298|0299|0300|0301|0302|0303|0304|0305|0306|0307|0308|0309|0311|0312|0313|0314|0315|03